# iForest hyperparam selection

In [2]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [3]:
import time
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

from sdss.metadata import MetaData

meta = MetaData()

# Custom Functions

## Winner iForest

In [4]:
def pick_iforest_params(df, consensus_threshold):

    """
    Given a dataframe of boolean anomaly flags from different
    Isolation Forest models (with different hyperparameters),
    calculate which parameter set is the most stable, i.e. which
    parameter set's anomalies overlap the most with the
    consensus anomalies (those that at least `consensus_threshold`
    models agreed were anomalies).
    """

    assert consensus_threshold > 0
    assert consensus_threshold <= df.shape[1] - 1

    stability_scores = {}

    # Identify points that at least N of the models agreed were anomalies
    consensus_anomalies = df['consensus_score'] >= consensus_threshold

    for col in df.columns[:-1]: # Exclude the consensus_score column
        # How many of this model's anomalies are also 'consensus' anomalies?
        overlap = (df[col] & consensus_anomalies).sum()
        stability_scores[col] = overlap

    # The "Optimal" params are the ones with the highest overlap
    best_params = max(stability_scores, key=stability_scores.get)

    print(f"The most stable parameter set is: {best_params}")

    return stability_scores, best_params

## Stability scores 

In [5]:
def compute_stability_score(grid_pred_df, stab_scores_dict, consensus_threshold=19):

    consensus_anomalies = grid_pred_df['consensus_score'] >= consensus_threshold
    n_consensus = consensus_anomalies.sum()

    print(f"N consensus: {n_consensus}")

    stab_scores_df = pd.DataFrame.from_dict(stab_scores_dict, orient='index')

    stab_scores_df.sort_values(by=0, ascending=False, inplace=True)
    stab_scores_df.columns = ['n_overlap']
    stab_scores_df['pct_overlap'] = stab_scores_df / n_consensus*100

    samples = []
    estimators = []
    features = []

    for name in stab_scores_df.index.to_list():

        splitted = name.split("_")

        s = int(splitted[0][1:])
        samples.append(s)

        e = int(splitted[1][1:])
        estimators.append(e)

        f = int(splitted[2][1:])
        features.append(f)

    stab_scores_df['samples'] = samples
    stab_scores_df['estimators'] = estimators
    stab_scores_df['features'] = features

    return stab_scores_df[['estimators', 'samples', 'features', 'n_overlap', 'pct_overlap']]


## Scale data

In [6]:
def standard_scaler(latent_arr):

    scaler = StandardScaler()
    
    latent_scaled = scaler.fit_transform(latent_arr)
    
    return latent_scaled

# Config

## Directories

In [7]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
ch4_dir = f"{thesis_dir}/chapters/04_figures"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
models_dir = f"{data_dir}/models"
latent_dir = f"{data_dir}/latent"
bins_ids = [f'bin_{i:02d}' for i in range(4)] 

## Data

In [8]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1
n_wave = wave.shape

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)


## Latent per bin

In [9]:
latent_bin_dict = {}

for bin_id in bins_ids:

    latent_bin_dict[bin_id] = np.load(
        f"{latent_dir}/{bin_id}/latent_{bin_id}.npy"
    )

# Consensus based stability

In [10]:
iforest_hyper_params_df_dict = {}

for bin_id in bins_ids:

    print(
        "Loading Isolation Forest hyperparameter"
        f"search results for {bin_id}")
        
    iforest_hyper_params_df_dict[bin_id] = pd.read_csv(
        f"{latent_dir}/{bin_id}/iforest/"
        f"iforest_hypersearch_{bin_id}.csv",
    )

Loading Isolation Forest hyperparametersearch results for bin_00
Loading Isolation Forest hyperparametersearch results for bin_01
Loading Isolation Forest hyperparametersearch results for bin_02
Loading Isolation Forest hyperparametersearch results for bin_03


In [11]:
bin_id = 'bin_03'
iforest_hyper_params_df_dict[bin_id].head()

,s64_e100_f100,s64_e100_f75,s64_e100_f50,s64_e200_f100,s64_e200_f75,s64_e200_f50,s64_e300_f100,s64_e300_f75,s64_e300_f50,s128_e100_f100,...,s512_e100_f100,s512_e100_f75,s512_e100_f50,s512_e200_f100,s512_e200_f75,s512_e200_f50,s512_e300_f100,s512_e300_f75,s512_e300_f50,consensus_score
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
stability_scores_dict = {}
best_params_dict = {}
# there are 36 models
consensus_threshold = 19

for bin_id, df in iforest_hyper_params_df_dict.items():

    print(f"Calculating stability scores for {bin_id}")

    stability_scores, best_params = pick_iforest_params(
        df=df.copy(),
        consensus_threshold=consensus_threshold
    )

    stability_scores_dict[bin_id] = stability_scores
    best_params_dict[bin_id] = best_params

Calculating stability scores for bin_00
The most stable parameter set is: s256_e300_f100
Calculating stability scores for bin_01
The most stable parameter set is: s256_e300_f75
Calculating stability scores for bin_02
The most stable parameter set is: s256_e300_f100
Calculating stability scores for bin_03
The most stable parameter set is: s128_e300_f100


In [13]:
best_params_dict

{'bin_00': 's256_e300_f100',
 'bin_01': 's256_e300_f75',
 'bin_02': 's256_e300_f100',
 'bin_03': 's128_e300_f100'}

# Bin 4

In [14]:
bin_id = 'bin_03'
consensus_03_df = iforest_hyper_params_df_dict[bin_id].copy()
consensus_03_df.head(3)

,s64_e100_f100,s64_e100_f75,s64_e100_f50,s64_e200_f100,s64_e200_f75,s64_e200_f50,s64_e300_f100,s64_e300_f75,s64_e300_f50,s128_e100_f100,...,s512_e100_f100,s512_e100_f75,s512_e100_f50,s512_e200_f100,s512_e200_f75,s512_e200_f50,s512_e300_f100,s512_e300_f75,s512_e300_f50,consensus_score
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
consensus_threshold = 19
consensus_mask = consensus_03_df['consensus_score'] >= consensus_threshold
consensus_03_df.loc[consensus_mask, ['s128_e300_f100', 'consensus_score']].sum(axis=0)
consensus_mask.sum()

1629

In [16]:
bin_id = 'bin_03'
stab_scores_03_dict = stability_scores_dict[bin_id]
grid_pred_03_df = iforest_hyper_params_df_dict[bin_id]

stab_scores_03_df = compute_stability_score(
    grid_pred_df=grid_pred_03_df,
    stab_scores_dict=stab_scores_03_dict,
    consensus_threshold=19
)
stab_scores_03_df.head(18)

N consensus: 1629


,estimators,samples,features,n_overlap,pct_overlap
s128_e300_f100,300,128,100,1543,94.720688
s512_e300_f50,300,512,50,1542,94.659300
s256_e300_f50,300,256,50,1539,94.475138
s256_e300_f100,300,256,100,1533,94.106814
s256_e300_f75,300,256,75,1526,93.677103
s512_e300_f75,300,512,75,1519,93.247391
s128_e300_f50,300,128,50,1516,93.063229
s256_e200_f75,200,256,75,1514,92.940454
s256_e200_f100,200,256,100,1514,92.940454
s512_e200_f50,200,512,50,1508,92.572130


## Train and save model

In [17]:
bin_id = 'bin_03'
latent_scaled = standard_scaler(
    latent_bin_dict[bin_id]
)

e, s, f = best_params_dict[bin_id].split('_')
e, s, f = int(e[1:]), int(s[1:]), int(f[1:])/100

model_fname = f"iforest_{bin_id}_{best_params_dict[bin_id]}.joblib"

save_dir = os.path.join(
    latent_dir, bin_id, 'iforest'
)
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, model_fname)

start_time = time.perf_counter()

iforest = IsolationForest(
    n_estimators=e,
    max_samples=s,
    max_features=f,
    contamination=0.01,
    n_jobs=-1,
    random_state=42
)

iforest.fit(latent_scaled)

end_time = time.perf_counter()

print(
    f"Trained Isolation Forest model for {bin_id} "
    f"in {end_time - start_time:.2f} seconds."
)

joblib.dump(iforest, save_path)

Trained Isolation Forest model for bin_03 in 1.24 seconds.


['/home/elom/phd/code/latent/bin_03/iforest/iforest_bin_03_s128_e300_f100.joblib']

In [18]:
best_params_dict

{'bin_00': 's256_e300_f100',
 'bin_01': 's256_e300_f75',
 'bin_02': 's256_e300_f100',
 'bin_03': 's128_e300_f100'}

# Bin 3

In [19]:
bin_id = 'bin_02'
consensus_threshold = 19
df = iforest_hyper_params_df_dict[bin_id].copy()
consensus_mask = df['consensus_score'] >= consensus_threshold
df.loc[consensus_mask, ['s256_e300_f100', 'consensus_score']].sum()
consensus_mask.sum()


1782

In [20]:
bin_id = 'bin_02'
stab_scores_02_dict = stability_scores_dict[bin_id]
grid_pred_02_df = iforest_hyper_params_df_dict[bin_id]

stab_scores_02_df = compute_stability_score(
    grid_pred_df=grid_pred_02_df,
    stab_scores_dict=stab_scores_02_dict,
    consensus_threshold=19
)
stab_scores_02_df.head(15)

N consensus: 1782


,estimators,samples,features,n_overlap,pct_overlap
s256_e300_f100,300,256,100,1725,96.801347
s256_e300_f50,300,256,50,1722,96.632997
s128_e300_f75,300,128,75,1712,96.071829
s512_e300_f75,300,512,75,1711,96.015713
s128_e300_f100,300,128,100,1710,95.959596
s128_e200_f75,200,128,75,1709,95.903479
s256_e200_f100,200,256,100,1709,95.903479
s512_e300_f50,300,512,50,1706,95.735129
s512_e200_f75,200,512,75,1705,95.679012
s512_e200_f100,200,512,100,1703,95.566779


## Train and save models

In [21]:
bin_id = 'bin_02'
latent_scaled = standard_scaler(
    latent_bin_dict[bin_id]
)

e, s, f = best_params_dict[bin_id].split('_')
e, s, f = int(e[1:]), int(s[1:]), int(f[1:])/100

model_fname = f"iforest_{bin_id}_{best_params_dict[bin_id]}.joblib"

save_dir = os.path.join(
    latent_dir, bin_id, 'iforest'
)
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, model_fname)

start_time = time.perf_counter()

iforest = IsolationForest(
    n_estimators=e,
    max_samples=s,
    max_features=f,
    contamination=0.01,
    n_jobs=-1,
    random_state=42
)

iforest.fit(latent_scaled)

end_time = time.perf_counter()

print(
    f"Trained Isolation Forest model for {bin_id} "
    f"in {end_time - start_time:.2f} seconds."
)

joblib.dump(iforest, save_path)

Trained Isolation Forest model for bin_02 in 2.24 seconds.


['/home/elom/phd/code/latent/bin_02/iforest/iforest_bin_02_s256_e300_f100.joblib']

# Bin 2

In [22]:
bin_id = 'bin_01'
consensus_threshold = 19
df = iforest_hyper_params_df_dict[bin_id].copy()
consensus_mask = df['consensus_score'] >= consensus_threshold
df.loc[consensus_mask, ['s256_e300_f75', 'consensus_score']].sum()
consensus_mask.sum()

1739

In [23]:
bin_id = 'bin_01'
stab_scores_01_dict = stability_scores_dict[bin_id]
grid_pred_01_df = iforest_hyper_params_df_dict[bin_id]

stab_scores_01_df = compute_stability_score(
    grid_pred_df=grid_pred_01_df,
    stab_scores_dict=stab_scores_01_dict,
    consensus_threshold=19
)
stab_scores_01_df.head(10)

N consensus: 1739


,estimators,samples,features,n_overlap,pct_overlap
s256_e300_f75,300,256,75,1667,95.859689
s256_e300_f50,300,256,50,1659,95.399655
s256_e200_f75,200,256,75,1650,94.882116
s256_e200_f100,200,256,100,1649,94.824612
s256_e300_f100,300,256,100,1646,94.652099
s128_e300_f75,300,128,75,1640,94.307073
s128_e300_f100,300,128,100,1635,94.019551
s512_e300_f50,300,512,50,1635,94.019551
s128_e300_f50,300,128,50,1631,93.789534
s256_e200_f50,200,256,50,1630,93.732030


## Train and save model

In [24]:
bin_id = 'bin_01'
latent_scaled = standard_scaler(
    latent_bin_dict[bin_id]
)

e, s, f = best_params_dict[bin_id].split('_')
e, s, f = int(e[1:]), int(s[1:]), int(f[1:])/100

model_fname = f"iforest_{bin_id}_{best_params_dict[bin_id]}.joblib"

save_dir = os.path.join(
    latent_dir, bin_id, 'iforest'
)
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, model_fname)

start_time = time.perf_counter()

iforest = IsolationForest(
    n_estimators=e,
    max_samples=s,
    max_features=f,
    contamination=0.01,
    n_jobs=-1,
    random_state=42
)

iforest.fit(latent_scaled)

end_time = time.perf_counter()

print(
    f"Trained Isolation Forest model for {bin_id} "
    f"in {end_time - start_time:.2f} seconds."
)

joblib.dump(iforest, save_path)

Trained Isolation Forest model for bin_01 in 2.98 seconds.


['/home/elom/phd/code/latent/bin_01/iforest/iforest_bin_01_s256_e300_f75.joblib']

# Bin 1

In [25]:
bin_id = 'bin_00'
consensus_threshold = 19
df = iforest_hyper_params_df_dict[bin_id].copy()
consensus_mask = df['consensus_score'] >= consensus_threshold
df.loc[consensus_mask, ['s256_e300_f100', 'consensus_score']].sum()
consensus_mask.sum()

1767

In [26]:
bin_id = 'bin_00'
stab_scores_00_dict = stability_scores_dict[bin_id]
grid_pred_00_df = iforest_hyper_params_df_dict[bin_id]

stab_scores_00_df = compute_stability_score(
    grid_pred_df=grid_pred_00_df,
    stab_scores_dict=stab_scores_00_dict,
    consensus_threshold=19
)
stab_scores_00_df.head(14)

N consensus: 1767


,estimators,samples,features,n_overlap,pct_overlap
s256_e300_f100,300,256,100,1736,98.245614
s128_e300_f50,300,128,50,1730,97.906055
s256_e300_f50,300,256,50,1721,97.396718
s256_e200_f100,200,256,100,1713,96.943973
s256_e300_f75,300,256,75,1708,96.661007
s128_e300_f100,300,128,100,1706,96.547821
s128_e200_f50,200,128,50,1702,96.321449
s256_e200_f75,200,256,75,1700,96.208263
s128_e300_f75,300,128,75,1698,96.095076
s512_e300_f100,300,512,100,1698,96.095076


In [27]:
bin_id = 'bin_00'
latent_scaled = standard_scaler(
    latent_bin_dict[bin_id]
)

e, s, f = best_params_dict[bin_id].split('_')
e, s, f = int(e[1:]), int(s[1:]), int(f[1:])/100

model_fname = f"iforest_{bin_id}_{best_params_dict[bin_id]}.joblib"

save_dir = os.path.join(
    latent_dir, bin_id, 'iforest'
)
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, model_fname)

start_time = time.perf_counter()

iforest = IsolationForest(
    n_estimators=e,
    max_samples=s,
    max_features=f,
    contamination=0.01,
    n_jobs=-1,
    random_state=42
)

iforest.fit(latent_scaled)

end_time = time.perf_counter()

print(
    f"Trained Isolation Forest model for {bin_id} "
    f"in {end_time - start_time:.2f} seconds."
)

joblib.dump(iforest, save_path)

Trained Isolation Forest model for bin_00 in 2.18 seconds.


['/home/elom/phd/code/latent/bin_00/iforest/iforest_bin_00_s256_e300_f100.joblib']